In [1]:
import pandas as pd
from pathlib import Path

file_name = "C:\\Users\\nnish\\OneDrive\\Desktop\\bank_customer_churn_prediction_dataset.xls"
data_path = Path(file_name)

df = pd.read_csv(data_path)
print("Dataset loaded successfully!")
df.head()

Dataset loaded successfully!


,Customer_ID,Age,Account_Type,Account_Balance,Monthly_Transactions,Number_of_Products,Mobile_App_User,Customer_Service_Calls,Months_Inactive,Credit_Card_Holder,Satisfaction_Score,Branch_Visits,Customer_Churned
0,1,26.0,Basic,155808.0,107,5,No,2,15,No,5.0,3,No
1,2,71.0,Basic,186299.0,109,4,Yes,4,10,Yes,2.0,7,No
2,3,32.0,Basic,102173.0,71,5,Yes,0,7,Yes,3.0,7,No
3,4,52.0,Premium,194319.0,19,5,Yes,6,12,Yes,2.0,0,Yes
4,5,28.0,Premium,127968.0,12,3,No,11,15,Yes,2.0,18,No


In [3]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (387, 13)
Columns: ['Customer_ID', 'Age', 'Account_Type', 'Account_Balance', 'Monthly_Transactions', 'Number_of_Products', 'Mobile_App_User', 'Customer_Service_Calls', 'Months_Inactive', 'Credit_Card_Holder', 'Satisfaction_Score', 'Branch_Visits', 'Customer_Churned']


In [4]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

Missing values:
Customer_ID               0
Age                       7
Account_Type              7
Account_Balance           7
Monthly_Transactions      0
Number_of_Products        0
Mobile_App_User           7
Customer_Service_Calls    0
Months_Inactive           0
Credit_Card_Holder        7
Satisfaction_Score        7
Branch_Visits             0
Customer_Churned          0
dtype: int64

Duplicates: 7


In [5]:
df = df.drop_duplicates()
df = df.drop(columns=['Customer_ID'])

print("Cleaned! New shape:", df.shape)

Cleaned! New shape: (380, 12)


In [9]:
business_problem = "A bank wants to identify customers who are at risk of closing their accounts."
ml_problem       = "This is a binary classification problem where the output is either 0 (Stay) or 1 (Churn)."
prediction_goal  = "Predict whether a bank customer will leave or stay based on their account activity."

print("Business Problem:", business_problem)
print("ML Problem:",       ml_problem)
print("Prediction Goal:",  prediction_goal)

Business Problem: A bank wants to identify customers who are at risk of closing their accounts.
ML Problem: This is a binary classification problem where the output is either 0 (Stay) or 1 (Churn).
Prediction Goal: Predict whether a bank customer will leave or stay based on their account activity.


In [10]:
target_column = "Customer_Churned"

print("Target column:", target_column)
print("\nTarget values:")
print(df[target_column].value_counts())

Target column: Customer_Churned

Target values:
Customer_Churned
No     339
Yes     41
Name: count, dtype: int64


In [12]:
feature_columns = [
    "Age",
    "Account_Type",
    "Account_Balance",
    "Monthly_Transactions",
    "Number_of_Products",
    "Mobile_App_User",
    "Customer_Service_Calls",
    "Months_Inactive",
    "Credit_Card_Holder",
    "Satisfaction_Score",
    "Branch_Visits"
]

X = df[feature_columns]
y = df[target_column]

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (380, 11)
Shape of y: (380,)


In [15]:
df_clean = df.copy()
df_clean = df_clean.drop_duplicates()
print("Shape after removing duplicates:", df_clean.shape)

numeric_columns = ["Age", "Account_Balance", "Satisfaction_Score"]
for column in numeric_columns:
    median_value = df_clean[column].median()
    df_clean[column] = df_clean[column].fillna(median_value)

categorical_columns = ["Account_Type", "Mobile_App_User", "Credit_Card_Holder"]
for column in categorical_columns:
    most_frequent_value = df_clean[column].mode()[0]
    df_clean[column] = df_clean[column].fillna(most_frequent_value)

print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())

Shape after removing duplicates: (380, 12)

Missing values after cleaning:
Age                       0
Account_Type              0
Account_Balance           0
Monthly_Transactions      0
Number_of_Products        0
Mobile_App_User           0
Customer_Service_Calls    0
Months_Inactive           0
Credit_Card_Holder        0
Satisfaction_Score        0
Branch_Visits             0
Customer_Churned          0
dtype: int64


In [16]:
X = df_clean[feature_columns]
y = df_clean[target_column]

y = y.map({"No": 0, "Yes": 1})

print("Target value counts after conversion:")
print(y.value_counts())

Target value counts after conversion:
Customer_Churned
0    339
1     41
Name: count, dtype: int64


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:",  X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:",  y_test.shape)

Shape of X_train: (304, 11)
Shape of X_test: (76, 11)
Shape of y_train: (304,)
Shape of y_test: (76,)


In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numerical_features = [
    "Age", "Account_Balance", "Monthly_Transactions",
    "Number_of_Products", "Customer_Service_Calls",
    "Months_Inactive", "Satisfaction_Score", "Branch_Visits"
]

categorical_features = [
    "Account_Type", "Mobile_App_User", "Credit_Card_Holder"
]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed  = preprocessor.transform(X_test)

print("X_train preprocessing completed!")
print("X_test preprocessing completed!")

X_train preprocessing completed!
X_test preprocessing completed!


In [20]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_preprocessed, y_train)

print("Model training completed!")

Model training completed!


In [21]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test_preprocessed)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Not Churned", "Churned"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9078947368421053

Classification Report:
              precision    recall  f1-score   support

 Not Churned       0.92      0.99      0.95        68
     Churned       0.67      0.25      0.36         8

    accuracy                           0.91        76
   macro avg       0.79      0.62      0.66        76
weighted avg       0.89      0.91      0.89        76

Confusion Matrix:
[[67  1]
 [ 6  2]]


In [28]:
print("""
SUMMARY

What the model is trying to predict:
My model is trying to predict whether a bank customer will leave the bank or stay. This is called churn  prediction. The answer is either 
Yes (they leave) or No (they stay).

What dataset did I use?
I used the Bank Customer Churn Prediction Dataset, which contains information about 387 bank customers, including their age, account balance, 
satisfaction score and other banking behaviours.

What features and target do I select:
I used 11 features such as Age, Account Balance, Monthly Transactions, Satisfaction Score and Customer Service Calls to predict the target
which is Customer Churned (Yes or No).

What result do I obtain:
My Logistic Regression model was able to predict customer churn with reasonable accuracy. The model performed better at identifying customers who 
stay than customers who leave because there were fewer churn examples in the dataset.

One limitation:
The biggest limitation is that only 42 out of 387 customers actually churned. This means the dataset is imbalanced, and the model does not have enough
examples of churned customers to learn from properly. A larger and more balanced dataset would give much better results.
""")


SUMMARY

What the model is trying to predict:
My model is trying to predict whether a bank customer will leave the bank or stay. This is called churn  prediction. The answer is either 
Yes (they leave) or No (they stay).

What dataset I used:
I used the Bank Customer Churn Prediction Datasetwhich contains information about 387 bank customers including their age, account balance, 
satisfaction score and other banking behaviours.

What features and target I selected:
I used 11 features such as Age, Account Balance, Monthly Transactions, Satisfaction Score and Customer Service Calls to predict the target
which is Customer Churned (Yes or No).

What result I obtained:
My Logistic Regression model was able to predict customer churn with a reasonable accuracy. The model performed better at identifying customers who 
stay than customers who leave because there were fewer churn examples in the dataset.

One limitation:
The biggest limitation is that only 42 out of 387 customers actually churn